# Debug Notebook: Simulate Players Using API.py (FastAPI wrapper)

This notebook is the equivalent of `games/games_db_debug.ipynb`, but instead of calling the game engine directly it talks to the FastAPI server in `API.py` over HTTP + WebSocket — exactly like a future web client would.

**Prerequisite:** start the server first in another terminal:
```powershell
python API.py   # serves http://127.0.0.1:8000
```

Endpoints used here:
- `POST /create_game` → creates the game with p1, returns the `game_id` to share
- `POST /join_game/{game_id}` → registers p2 and initializes the game (hands dealt, planet rolled)
- `GET /game/{game_id}` → full state, no hidden info — used to refresh between actions
- `WS /ws/{game_id}/{player_name}` → send one JSON action at a time, receive the personalized state back

❗ Each player only receives a new state when **they** send a message, so after every action the loop refreshes the authoritative state via `GET /game/{id}` (same pattern as the direct-engine notebook).

In [15]:
import sys
import json
import random
import asyncio
from pathlib import Path

import requests
import websockets
import polars as pl
import nest_asyncio   # allow asyncio.run() inside the Jupyter event loop
nest_asyncio.apply()

# make the project root importable (models, player_ai)
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models import GameState, PlayerState
import player_ai.playerai as pai

BASE_URL = "http://127.0.0.1:8000"

In [2]:
class WSPlayer:
    """One websocket connection per player. Sends one JSON action at a time and receives the personalized game state back (opponent's hand/mana/deck hidden)."""

    def __init__(self, name, game_id):
        self.name = name
        self.game_id = game_id
        self.ws = None
        self.state = None   # last personalized GameState dict received from the server

    async def connect(self):
        uri = f"ws://127.0.0.1:8000/ws/{self.game_id}/{self.name}"
        self.ws = await websockets.connect(uri)
        self.state = json.loads(await self.ws.recv())   # initial personalized state

    async def send(self, message):
        """send a game action; returns True if the engine accepted it (full state returned), False on rejection"""
        await self.ws.send(json.dumps(message))
        resp = json.loads(await self.ws.recv())
        if 'players' not in resp:   # rejection / error response -> keep previous state
            print(f"    [{self.name}] server says: {resp.get('message')}")
            return False
        self.state = resp
        return True

    async def close(self):
        await self.ws.close()

In [3]:
CARDS_DB = pl.from_records(requests.get(f"{BASE_URL}/cardpool").json())
CARDS_DB.head(3)

faction,mana,advancing,shield,condition,effect,effect_number,rare,condeff_value,card_id,prompt,negative_prompt,name
str,i64,i64,i64,str,str,i64,bool,i64,str,str,str,str
"""Dwarves""",2,2,3,"""no_condition""","""advancing""",1,false,0,"""Dwa23_79c784""","""A small dark skin fat female …","""""","""Hanna Zanna"""
"""Dwarves""",2,1,3,"""no_condition""","""advancing""",2,false,-1,"""Dwa23_5cd6c5""","""A small fat female dwarf wea…","""""","""Filda Rurik"""
"""Dwarves""",2,0,3,"""no_condition""","""advancing""",3,false,-2,"""Dwa23_5a81c9""","""A small old slim female dwarf…","""""","""Korla Hildir"""


In [10]:
def make_deck():
    """Randomly choose a faction and select 15 cards from that faction (same as the direct-engine notebook)."""
    faction = random.choice(CARDS_DB['faction'].unique().to_list())
    filtered_cards = CARDS_DB.filter(pl.col('faction') == faction)
    return random.sample(filtered_cards['card_id'].to_list(), 15)

# p1 creates the game (POST /create_game)
p1_name, p2_name = 'Anne', 'Boris'
resp = requests.post(f"{BASE_URL}/create_game", json={"name": p1_name, "deck": make_deck()})
print("create response:", resp.status_code, resp.json())
GAME_ID = resp.json()['game_id']

# p2 joins the same game (POST /join_game/{game_id}) -> initializes the game
resp = requests.post(f"{BASE_URL}/join_game/{GAME_ID}", json={"name": p2_name, "deck": make_deck()})
print("join response:", resp.status_code)
join_state = GameState.model_validate(resp.json())
print(f"state after join: {join_state.state}")

create response: 200 {'success': True, 'game_id': '26_08_23_14_40_01_h7XWb', 'player_id': 'Anne'}
join response: 200
state after join: waiting for both players to put 3 cards in hand


In [11]:
# Connect both players to the game (one websocket each)
w1, w2 = WSPlayer(p1_name, GAME_ID), WSPlayer(p2_name, GAME_ID)

async def _connect_all():
    await w1.connect()
    await w2.connect()
asyncio.run(_connect_all())

# build local PlayerState + AI for each player from their personalized state
p1 = PlayerState.model_validate(w1.state['players'][p1_name])
p1_ai = pai.PlayerAI(p1)
p2 = PlayerState.model_validate(w2.state['players'][p2_name])
p2_ai = pai.PlayerAI(p2)

def full_refresh():
    """GET the full game state (no hidden info) and rebuild both local players + AIs from it.
     Needed because each player only receives a new state when THEY send a message."""
    global p1, p2, p1_ai, p2_ai
    gs = GameState.model_validate(requests.get(f"{BASE_URL}/game/{GAME_ID}").json())
    p1 = PlayerState.model_validate(gs.players[p1_name])
    p2 = PlayerState.model_validate(gs.players[p2_name])
    p1_ai.update_player_state(p1, oppo_state=p2, game=gs)
    p2_ai.update_player_state(p2, oppo_state=p1, game=gs)
    return gs

print(f"p1 hand: {p1.hand}")
print(f"p2 hand (hidden in w1.state): {w1.state['players'][p2_name]['hand']}")

p1 hand: ['Orc43_3d7982', 'Orc54_1eb9b0', 'Orc32_dd0cdf', 'Orc43_f56a33', 'Orc43_1b5c49', 'Orc54_f49e1d']
p2 hand (hidden in w1.state): []


At this point, both players are connected to the same game and they have (6) cards in hand. The state says "waiting for both players to put 3 cards in hand" — so each player puts 3 cards into mana through their websocket:

In [12]:
async def _initial_mana():
    for w, p_ai in [(w1, p1_ai), (w2, p2_ai)]:
        msg = {'cards': p_ai.put_mana(num_cards=3), 'to': 'mana', 'mode': '', 'pendings': []}
        ok = await w.send(msg)
        print(f"{w.name} put {msg['cards']} to mana -> state: {w.state['state'] if ok else '(rejected)'}")

asyncio.run(_initial_mana())

gs = full_refresh()
print(f"game state: {gs.state}")

Anne put ['Orc54_1eb9b0', 'Orc54_f49e1d', 'Orc43_1b5c49'] to mana -> state: waiting for both players to put 3 cards in hand
Boris put ['Twi33_4dde8f', 'Twi33_e5b024', 'Twi33_bfc0b9'] to mana -> state: turn 1 - waiting for first player (Boris) to play
game state: turn 1 - waiting for first player (Boris) to play


From now the game is initialized and every message has the same structure: `{'cards': [...], 'to': '...', 'mode': '...', 'pendings': []}`. Simulating a random simple game until it is over (same loop as the direct-engine notebook, but actions go through the websockets):

In [13]:
import re

_TURN_RE = re.compile(r"waiting for (?:first|second) player \((.+?)\) to play")
MAX_TURNS = 200   # safety guard: some random games can loop forever once all cards end up in the mana zones

async def _play_game():
    global gs
    gs = full_refresh()
    while 'game over' not in gs.state and gs.turn <= MAX_TURNS:
        print(f'Turn {gs.turn} --------------')
        print(f'\tgame state: {gs.state}')

        # Put mana phase if needed (both players act)
        if gs.state == "waiting for both players to mana or pass":
            for w, p_ai in [(w1, p1_ai), (w2, p2_ai)]:
                msg = p_ai.put_mana(num_cards=1, in_turn=True)
                ok = await w.send(msg)
                if msg['mode'] == 'pass':
                    print(f"{w.name} passed the mana phase")
                else:
                    print(f"{w.name} put {msg['cards'][0]} to mana" + ("" if ok else " (rejected)"))
            gs = full_refresh()
            print(f'\tgame state: {gs.state}')
            continue

        # Turn phase: only the expected player acts
        m = _TURN_RE.search(gs.state)
        if not m:
            print('unexpected state:', gs.state)
            break
        w, p_ai = (w1, p1_ai) if m.group(1) == p1_name else (w2, p2_ai)
        msg = p_ai.play_card()
        ok = await w.send(msg)
        print(f"\t\t{m.group(1)} played {msg['cards']} ({msg['mode']}) [{p_ai.player_state.hand}]")
        gs = full_refresh()
        if msg['mode'] == 'pass':
            p = PlayerState.model_validate(gs.players[m.group(1)])
            print(f"\t\t\tpassed - mana spent so far this turn: {p.mana_spend}/{len(p.mana)}")
        else:
            card_cost = CARDS_DB.filter(pl.col('card_id').is_in(msg['cards']))['mana'].sum()
            p = PlayerState.model_validate(gs.players[m.group(1)])
            print(f"\t\t\tplayed card cost {card_cost} mana - total spent this turn: {p.mana_spend}/{len(p.mana)}")
        print(f'\tgame state: {gs.state}')

    if 'game over' in gs.state:
        print(f"Game over - winner: {gs.winner}")
    else:
        print(f'Stopped at turn {gs.turn} (MAX_TURNS={MAX_TURNS}) - game did not end, see state above')

asyncio.run(_play_game())

Turn 1 --------------
	game state: turn 1 - waiting for first player (Boris) to play
		Boris played ['Twi22_541c8e'] (move) [['Twi22_790913', 'Twi22_541c8e', 'Twi22_f8986e']]
			played card cost 2 mana - total spent this turn: 2/3
	game state: turn 1 - waiting for second player (Anne) to play
Turn 1 --------------
	game state: turn 1 - waiting for second player (Anne) to play
		Anne played ['Orc32_dd0cdf'] (move) [['Orc43_3d7982', 'Orc32_dd0cdf', 'Orc43_f56a33']]
			played card cost 3 mana - total spent this turn: 3/3
	game state: turn 1 - waiting for first player (Boris) to play
Turn 1 --------------
	game state: turn 1 - waiting for first player (Boris) to play
		Boris played [] (pass) [['Twi22_790913', 'Twi22_f8986e']]
			passed - mana spent so far this turn: 2/3
	game state: turn 1 - waiting for second player (Anne) to play
Turn 1 --------------
	game state: turn 1 - waiting for second player (Anne) to play
		Anne played [] (pass) [['Orc43_3d7982', 'Orc43_f56a33']]
			passed - mana

In [14]:
async def _cleanup():
    await w1.close()
    await w2.close()
asyncio.run(_cleanup())

# Edge cases: invalid game id, unknown player on websocket, out-of-turn message
print("GET /game/unknown_game ->", requests.get(f"{BASE_URL}/game/unknown_game").status_code)
print("POST /create_game empty deck ->", requests.post(f"{BASE_URL}/create_game", json={"name": "X", "deck": []}).status_code)

async def _edge_ws():
    # unknown player: server should reject the connection with an error message
    bad = WSPlayer('NotAPlayer', GAME_ID)
    await bad.connect()
    print("unknown player initial state:", json.dumps(bad.state))
    await bad.close()

asyncio.run(_edge_ws())

GET /game/unknown_game -> 404
POST /create_game empty deck -> 400
unknown player initial state: {"success": false, "message": "player 'NotAPlayer' is not registered in game 26_08_23_14_40_01_h7XWb (use /create_game or /join_game first)"}
